**Part 1 — Primitive Types**

In [ ]:
# ── Integers ──────────────────────────────────────────────
order_id = 10045
quantity = 3
year = 2024

print(type(order_id))   # <class 'int'>

# ── Floats ────────────────────────────────────────────────
unit_price = 299.99
discount   = 0.15

# WARNING every DE must know this
print(0.1 + 0.2)        # 0.30000000000000004  ← not 0.3
# Floats are stored in binary — they cannot represent
# all decimals exactly. In financial pipelines this
# causes silent errors. We will handle this properly later.

# ── Strings ───────────────────────────────────────────────
customer_name  = "Ravi Sharma"
status         = 'active'
country_code   = "IN"

# ── Booleans ──────────────────────────────────────────────
is_paid     = True
is_returned = False

print(type(is_paid))    # <class 'bool'>

# In DE, booleans come from conditions constantly
# Example: a pipeline flag
pipeline_enabled = True

<class 'int'>
0.30000000000000004
<class 'bool'>


**Part 2 — The None Type**
This is one of the most important types in Data Engineering. More important than most tutorials make it.

In [ ]:
# None means the absence of a value
# In databases this maps to NULL
# In pipelines, unhandled Nones cause crashes

customer_email = None
delivery_date  = None

print(customer_email)         # None
print(type(customer_email))   # <class 'NoneType'>

# The dangerous mistake every beginner makes
name = None
# print(name.upper())         # AttributeError — crashes pipeline
                               # Never call methods on None directly

# The correct way — always check first
if name is not None:
    print(name.upper())
else:
    print("name is missing")

None
<class 'NoneType'>
name is missing


**Part 3 — Collection Types**
These four are the backbone of every data pipeline.

In [ ]:
# ── List — ordered, allows duplicates ─────────────────────
# Use when: you have a sequence of items, order matters

product_ids = [101, 102, 103, 101, 104]
# Notice 101 appears twice — lists allow duplicates
print(len(product_ids))      # 5
print(product_ids[0])        # 101  ← zero indexed
print(product_ids[-1])       # 104  ← last item

# ── Tuple — ordered, immutable ────────────────────────────
# Use when: data should not change after creation
# In DE: column definitions, config pairs, DB connection params

db_config = ("localhost", 5432, "tradesphere")
host, port, dbname = db_config    # unpacking
print(host)    # localhost
print(port)    # 5432

# You cannot do this to a tuple:
# db_config[0] = "production"    # TypeError — intentional

# ── Dictionary — key-value pairs ──────────────────────────
# Use when: you need to look something up by name
# In DE: configs, JSON records, API responses, row data

customer = {
    "customer_id"   : 5001,
    "name"          : "Priya Mehta",
    "country"       : "India",
    "is_premium"    : True,
    "total_orders"  : 12
}

print(customer["name"])              # Priya Mehta
print(customer.get("email", "N/A"))  # N/A — safe access
# Never use customer["email"] if the key might not exist
# It will crash. Always use .get() with a default

# ── Set — unordered, no duplicates ────────────────────────
# Use when: you need unique values only
# In DE: deduplication, membership checks

order_statuses = {"pending", "shipped", "delivered", "shipped"}
print(order_statuses)   # {'pending', 'shipped', 'delivered'}
# 'shipped' appears once — sets remove duplicates automatically

5
101
104
localhost
5432
Priya Mehta
N/A
{'pending', 'delivered', 'shipped'}


**Part 4 — Type Conversion**
Pipelines constantly receive data in the wrong type. This is how you handle it.

In [ ]:
# Data comes in from CSV files, APIs, databases
# It often arrives as strings even when it should be numbers

raw_quantity   = "5"        # came from a CSV
raw_price      = "1299.50"  # came from an API response
raw_flag       = "True"     # came from a config file

# Convert properly
quantity = int(raw_quantity)
price    = float(raw_price)

# String "True" is NOT a boolean True — common DE mistake
flag_wrong = bool("False")   # True  ← WRONG, non-empty string is True
flag_right = raw_flag == "True"  # True  ← correct way

print(quantity)      # 5
print(price)         # 1299.5
print(flag_wrong)    # True   ← dangerous
print(flag_right)    # True   ← correct

# Always validate before converting
raw_value = "not_a_number"
try:
    converted = int(raw_value)
except ValueError:
    print(f"Cannot convert '{raw_value}' to int — will use default 0")
    converted = 0

5
1299.5
True
True
Cannot convert 'not_a_number' to int — will use default 0


**Task:** You are ingesting a raw customer record from a CSV file. Everything arrives as a string. Create the following and fix the types:

In [ ]:
raw_record = {
    "customer_id"  : "3021",
    "name"         : "Arjun Patel",
    "age"          : "28",
    "total_spent"  : "45230.75",
    "is_active"    : "True",
    "referral_code": "None"
}

cleaned_record = {
    "customer_id"  : int(raw_record["customer_id"]),
    "name"         : raw_record["name"],
    "age"          : int(raw_record["age"]),
    "total_spent"  : float(raw_record["total_spent"]),
    "is_active"    : raw_record["is_active"].strip().lower() == "true",
    "referral_code": None if raw_record["referral_code"] == "None" else raw_record["referral_code"]
}

for key, value in cleaned_record.items():
    print(f"{key}: {value} ({type(value).__name__})")

customer_id: 3021 (int)
name: Arjun Patel (str)
age: 28 (int)
total_spent: 45230.75 (float)
is_active: True (bool)
referral_code: None (NoneType)


In [ ]:
test_values = ["True", "False", "true", "false", "TRUE", "  True  ", "  false  "]

for val in test_values:
  result = val.strip().lower() == "true"

  print(f"'{val}' → {result} ({type(result).__name__})")

'True' → True (bool)
'False' → False (bool)
'true' → True (bool)
'false' → False (bool)
'TRUE' → True (bool)
'  True  ' → True (bool)
'  false  ' → False (bool)


**Section 1.2 — Operators. Let's go.**

**Why operators matter in DE specifically**

Every filter you write in a pipeline is an operator. Every business rule — "flag orders above ₹10,000" or "skip records where status is null" — is a combination of operators. Get comfortable with these and conditional logic becomes effortless.

In [ ]:
#Arithmetic Operators
#Just one line to start. Run this in Colab:
revenue = 15000
tax_rate = 0.18

tax = revenue * tax_rate
print(tax)

print(17 // 3)   # 5  ← floor division, drops the decimal
print(17 % 3)    # 2  ← remainder, called modulo

#Why does modulo matter in DE?
#Run this and think about what it does:
for i in range(1, 11):
    if i % 2 == 0:
        print(f"Row {i} — process this batch")


2700.0
5
2
Row 2 — process this batch
Row 4 — process this batch
Row 6 — process this batch
Row 8 — process this batch
Row 10 — process this batch


Write a single expression that calculates the discounted price of an item.

original_price = 8500

discount_percent = 12

# Your one line here
# final_price should be 7480.0

In [ ]:
original_price = 8500

discount_percent = 12

final_price = original_price * (1 - discount_percent / 100)
print(final_price)

7480.0


 A record coming into your pipeline

record = {
    "order_id"     : 5021,
    "amount"       : 4500,
    "status"       : "processing",
    "is_verified"  : True
}

valid_statuses = {"pending", "shipped", "delivered", "cancelled", "processing"}

 Write conditions that print:
 1. Whether the order qualifies for free shipping (amount > 3000 and is_verified)
 2. Whether the status is valid
 3. Whether the order needs a review (amount > 10000 or not is_verified)

In [ ]:
record = {
    "order_id"     : 5021,
    "amount"       : 4500,
    "status"       : "processing",
    "is_verified"  : True
}

valid_statuses = {"pending", "shipped", "delivered", "cancelled", "processing"}

#1. Whether the order qualifies for free shipping (amount > 3000 and is_verified)
print("Free Shiping:", record['amount'] > 3000 and record['is_verified'])

#2. Whether the status is valid
print("Valid:", record["status"] in valid_statuses)

#3. Whether the order needs a review (amount > 10000 or not is_verified)
print("Needs Review:", record["amount"] > 10000 or not record['is_verified'])

Free Shiping: True
Valid: True
Needs Review: False


In [ ]:
print(0 or False)
print(False or 0)

False
0


Write an if/elif/else block that categorises a customer:

Rules:
spent > 50000 → "Platinum"

spent > 20000 → "Gold"  

spent > 5000  → "Silver"

anything else → "Bronze"


In [ ]:
total_spent = 32000

if total_spent > 50000:
  print("Platinum")
elif total_spent > 20000:
  print("Gold")
elif total_spent > 5000:
  print("Silver")
else:
  print("Bronze")


Gold


Now make it reusable
Right now your code only works for one hardcoded value. In a real pipeline you have thousands of customers.

Wrap it in a function:

In [ ]:
def categorised_customer(total_spent):
  if total_spent > 50000:
    return "Platinum"
  elif total_spent > 20000:
    return "Gold"
  elif total_spent > 5000:
    return "Silver"
  else:
    return "Bronze"


In [ ]:
customers = [72000, 32000, 8000, 1500]

for amount in customers:
  tier = categorised_customer(amount)
  print(f"Spent: {amount} → {tier}")

Spent: 72000 → Platinum
Spent: 32000 → Gold
Spent: 8000 → Silver
Spent: 1500 → Bronze


In [ ]:
orders = [
    {"order_id": 1, "amount": 4500, "status": "delivered"},
    {"order_id": 2, "amount": 12000, "status": "pending"},
    {"order_id": 3, "amount": 800, "status": "delivered"},
]

for order in orders:
    print(order["order_id"], order["amount"])

1 4500
2 12000
3 800


Now one small addition — **enumerate** gives you the index too:

In [ ]:
for index, order in enumerate(orders):
  print(f"Row {index}: order{order['order_id']}")

Row 0: order1
Row 1: order2
Row 2: order3


In [ ]:
import time

def fetch_data(attempt):
    if attempt < 3:
        raise ConnectionError("API not responding")
    return {"records": 100}

max_retries = 5
attempt = 0

while attempt < max_retries:
    try:
        result = fetch_data(attempt)
        print(f"Success on attempt {attempt + 1}")
        break
    except ConnectionError as e:
        attempt += 1
        print(f"Attempt {attempt} failed — retrying...")

Attempt 1 failed — retrying...
Attempt 2 failed — retrying...
Attempt 3 failed — retrying...
Success on attempt 4


One task combining for, if, and your categorise_customer function:

Loop through all customers
 For each one print:
 "Priya → Platinum" etc.

 AND separately print how many are Gold or above

Two things — the loop with tiers, and a count of Gold or above. Paste your attempt.

In [ ]:
customers = [
    {"name": "Priya",  "total_spent": 62000},
    {"name": "Arjun",  "total_spent": 25000},
    {"name": "Sneha",  "total_spent": 4200},
    {"name": "Rahul",  "total_spent": 180000},
    {"name": "Divya",  "total_spent": 9500},
]

def categorised_customer(total_spent):
  if total_spent > 50000:
    return "Platinum"
  elif total_spent > 20000:
    return "Gold"
  elif total_spent > 5000:
    return "Silver"
  else:
    return "Bronze"

gold_or_above_count = 0

for amount in customers:
  tier = categorised_customer(amount["total_spent"])

  print(f"{amount['name']} -> {tier}")

  if tier in ("Gold", "Platinum"):
    gold_or_above_count += 1
print("Gold or above customers:", gold_or_above_count)

Priya -> Platinum
Arjun -> Gold
Sneha -> Bronze
Rahul -> Platinum
Divya -> Silver
Gold or above customers: 3


Comprehensions — the DE shortcut
You just wrote a loop to process a list.

Python has a cleaner one-liner for this called a list comprehension.

In [ ]:
tier = [categorised_customer(c["total_spent"]) for c in customers]
print(tier)

['Platinum', 'Gold', 'Bronze', 'Platinum', 'Silver']


Now filtering with a comprehension:

In [ ]:
# Only customers who are Gold or above
top_customers = [c["name"] for c in customers if categorised_customer(c["total_spent"]) in ("Gold", "Platinum")]
print(top_customers)

['Priya', 'Arjun', 'Rahul']


In [ ]:
# Build a lookup: name → tier

tier_map = {c["name"]: categorised_customer(c["total_spent"]) for c in customers}
print(tier_map)

{'Priya': 'Platinum', 'Arjun': 'Gold', 'Sneha': 'Bronze', 'Rahul': 'Platinum', 'Divya': 'Silver'}


one comprehension exercise
Using comprehensions only — no for loops:
1. A list of amounts for delivered orders only
2. A dict of order_id → amount for all orders

In [ ]:
orders = [
    {"order_id": 101, "amount": 4500,  "status": "delivered"},
    {"order_id": 102, "amount": 12000, "status": "pending"},
    {"order_id": 103, "amount": 800,   "status": "delivered"},
    {"order_id": 104, "amount": 23000, "status": "cancelled"},
    {"order_id": 105, "amount": 6700,  "status": "delivered"},
]

delivered_only = [o["amount"] for o in orders if o["status"] == "delivered"]
print(delivered_only)

order_amount_map = {order['order_id']: order['amount'] for order in orders}
print(order_amount_map)

[4500, 800, 6700]
{101: 4500, 102: 12000, 103: 800, 104: 23000, 105: 6700}


In [ ]:
order_amount_map = {order['order_id']: order['amount'] for order in orders}
print(order_amount_map)

{101: 4500, 102: 12000, 103: 800, 104: 23000, 105: 6700}


Section 1.4 — Functions
This is the most important section in Module 1 for a Data Engineer.

Everything you build from here — pipelines, transformers, validators — lives inside functions. Let's make sure yours are built right.

**Pure vs Impure functions**


In [ ]:
# Impure function — depends on something outside itself
discount = 0.10

def calculate_price(amount):
    return amount * (1 - discount)

print(calculate_price(10000))

9000.0


In [ ]:
#Now the pure version:

def calculate_price(amount, discount):
  return amount * (1 - discount)

print(calculate_price(50000, 0.10))
print(calculate_price(100000, 0.20))

45000.0
80000.0


In [ ]:
#Default arguments

def calculate_price(amount, discount = 0.10):
  return amount * (1 - discount)

print(calculate_price(50000))
print(calculate_price(100000, 0.20))

45000.0
80000.0


Write a pure function called clean_record that takes a raw record dict and returns a cleaned version. It should:

clean_record should return:

customer_id  → int

name         → stripped of whitespace

total_spent  → float

is_active    → real boolean

country      → title case ("India")

In [ ]:
raw = {
    "customer_id" : "4021",
    "name"        : "  Ravi Sharma  ",
    "total_spent" : "28500.00",
    "is_active"   : "True",
    "country"     : "india"
}

def clean_record(raw):
  return{
      "customer_id" : int(raw["customer_id"]),
      "name" : raw["name"].strip(),
      "total_spent": float(raw["total_spent"]),
      "is_active": raw["is_active"].strip().lower() == "true",
      "country": raw["country"].strip().title()
  }

cleaned = clean_record(raw)
print(cleaned)

{'customer_id': 4021, 'name': 'Ravi Sharma', 'total_spent': 28500.0, 'is_active': True, 'country': 'India'}


In [ ]:
bad_record = {
    "customer_id" : "4022",
    "name"        : "Sneha Patel",
    "total_spent" : "15000.00",
    "is_active"   : "True"
    # country is missing
}

cleaned = clean_record(bad_record)

KeyError: 'country'

In [ ]:
#The fix — .get() with safe defaults:
def clean_record(raw):
    return {
        "customer_id" : int(raw["customer_id"]),
        "name"        : raw.get("name", "Unknown").strip(),
        "total_spent" : float(raw.get("total_spent", 0)),
        "is_active"   : raw.get("is_active", "false").strip().lower() == "true",
        "country"     : raw.get("country", "Unknown").strip().title()
    }

In [ ]:
bad_record = {
    "customer_id" : "4022",
    "name"        : "Sneha Patel",
    "total_spent" : "15000.00",
    "is_active"   : "True"
    # country is missing

}
cleaned = clean_record(bad_record)
print(cleaned)

{'customer_id': 4022, 'name': 'Sneha Patel', 'total_spent': 15000.0, 'is_active': True, 'country': 'Unknown'}


 ** *args and **kwargs**
These exist for one reason — flexibility. Sometimes you do not know in advance how many arguments a function will receive.
*args — variable number of positional arguments:

In [ ]:
def total_revenue(*amounts):
  return sum(amounts)

print(total_revenue(100,200,300))
print(total_revenue(1020,2030,3040,4050))

600
10140


**kwargs — variable number of keyword arguments:

This pattern is used in logging and config functions constantly.

In [ ]:
def log_event(event_type, **details):

  print(f"Event: {event_type}")
  for key, value in details.items():
    print(f" {key}: {value}")

log_event("Order_placed",
          order_id=5021,
          amount=12000,
          customer="Priya")

Event: Order_placed
 order_id: 5021
 amount: 12000
 customer: Priya


In [ ]:
def add_tag(record, tags=[]):
    tags.append("processed")
    record["tags"] = tags
    return record

r1 = add_tag({})
r2 = add_tag({})

print(r1["tags"])
print(r2["tags"])

['processed', 'processed']
['processed', 'processed']


Python creates the default value tags=[] once when the function is defined — not every time the function is called. That same list lives in memory permanently. Every call that uses the default is appending to the exact same list.
So what actually happened:

First call — appends "processed" to the shared list → ["processed"]
Second call — appends "processed" to the same list → ["processed", "processed"]
Both r1 and r2 point to that same list in memory

**The FIx:**

In [ ]:
def add_tag(record, tags=None):
    if tags is None:
        tags = []
    tags.append("processed")
    record["tags"] = tags
    return record

r1 = add_tag({})
r2 = add_tag({})

print(r1["tags"])
print(r2["tags"])

['processed']
['processed']


In [ ]:
total = 0

def process_orders(orders):
    for order in orders:
        total += order["amount"]
    return total

orders = [{"amount": 1000}, {"amount": 2000}]
print(process_orders(orders))

UnboundLocalError: cannot access local variable 'total' where it is not associated with a value

In [ ]:
def process_orders(orders, total=0):
    for order in orders:
        total += order["amount"]
    return total

print(process_orders(orders))

3000


**Section 1.5 — Error Handling**

This is the section that separates scripts from pipelines. A script that

crashes on one bad record is useless in production. Let's fix that.

In [ ]:
def process_record(raw_value):
    try:
        result = int(raw_value)
        print(f"Processed: {result}")
    except ValueError:
        print(f"Skipped bad value: {raw_value}")
    finally:
        print("Record attempt complete")

process_record("1500")
process_record("bad")

Processed: 1500
Record attempt complete
Skipped bad value: bad
Record attempt complete


Write a function called safe_clean that takes a raw record and returns a cleaned version. If any field fails conversion it should skip that record entirely and print which record failed and why — but never crash.

safe_clean(record) should:

→ return cleaned record if both fields convert successfully

→ return None and print the problem if anything fails

Then loop through records and collect only the successful ones

In [ ]:
def safe_clean(record):
  try:
    cleaned = {
        "customer_id": int(record["customer_id"]),
        "total_spent": float(record["total_spent"])
    }
    return cleaned
  except (ValueError, KeyError) as e:
    print(f"Skipped Record {record} -> Error {e}")

    return None

records = [
    {"customer_id": "301",  "total_spent": "15000.50"},
    {"customer_id": "302",  "total_spent": "not_a_number"},
    {"customer_id": "bad",  "total_spent": "9200.00"},
    {"customer_id": "304",  "total_spent": "42000.75"},
]

# Collect only valid records

cleaned_records = []

for record in records:
    cleaned = safe_clean(record)

    if cleaned:
        cleaned_records.append(cleaned)
        print(cleaned)   # only valid

print("Final Cleaned Records:", cleaned_records)

{'customer_id': 301, 'total_spent': 15000.5}
Skipped Record {'customer_id': '302', 'total_spent': 'not_a_number'} -> Error could not convert string to float: 'not_a_number'
Skipped Record {'customer_id': 'bad', 'total_spent': '9200.00'} -> Error invalid literal for int() with base 10: 'bad'
{'customer_id': 304, 'total_spent': 42000.75}
Final Cleaned Records: [{'customer_id': 301, 'total_spent': 15000.5}, {'customer_id': 304, 'total_spent': 42000.75}]


In [ ]:
#The production version
def safe_clean(record):
    try:
        return {
            "customer_id" : int(record["customer_id"]),
            "total_spent" : float(record["total_spent"])
        }
    except (ValueError, KeyError) as e:
        print(f"Skipped record {record.get('customer_id', 'unknown')} → {e}")
        return None
#Notice record.get('customer_id', 'unknown') in the error message —
#if customer_id itself is the missing key, your original code would throw
#another error inside the except block trying to print record['customer_id'].
#Always use .get() in error handlers.

**Section 1.6 — OOP for Data Engineering**
Before any code — the why.
In every pipeline you will ever build, you have three things — something that extracts data, something that transforms it, and something that loads it. Right now you would write three separate functions floating around in a file. OOP gives each of those a proper home — its own class, its own state, its own behaviour. That is why DE code is structured with classes.

**The simplest class first:**

In [ ]:
class Customer:
  def __init__(self, customer_id, name, total_spent):
    self.customer_id = customer_id
    self.name = name
    self.total_spent = total_spent
customer = Customer(3001, "Priya Mehta", 45000)
print(customer.customer_id)
print(customer.name)
print(customer.total_spent)


3001
Priya Mehta
45000


Now add behaviour

A class without methods is just a dictionary with extra steps. Methods are what make it useful:

In [ ]:
class Customer:
  def __init__(self, customer_id, name, total_spent):
    self.customer_id = customer_id
    self.name        = name
    self.total_spent = total_spent

  def get_tier(self):
    if self.total_spent > 50000:
      return "Platinum"
    elif self.total_spent > 20000:
      return "Gold"
    elif self.total_spent > 5000:
      return "Silver"
    else:
      return "Bronze"

  def apply_discount(self, percent):
    return self.total_spent * (1 - percent / 100)

customer = Customer(3001, "Priya Mehta", 45000)
print(customer.get_tier())
print(customer.apply_discount(10))

Gold
40500.0


In [ ]:
class Customer:
    def __init__(self, customer_id, name, total_spent):
        self.customer_id = customer_id
        self.name        = name
        self.total_spent = total_spent

    def __repr__(self):
        return f"Customer(id={self.customer_id}, name={self.name}, spent={self.total_spent})"

    def get_tier(self):
        if self.total_spent > 50000:
            return "Platinum"
        elif self.total_spent > 20000:
            return "Gold"
        elif self.total_spent > 5000:
            return "Silver"
        else:
            return "Bronze"

customer = Customer(3001, "Priya Mehta", 45000)
print(customer)

Customer(id=3001, name=Priya Mehta, spent=45000)


In [ ]:
class Order:
  VALID_STATUSES = {"pending", "shipped", "delivered", "cancelled"}

  def __init__(self, order_id, customer_id, amount, status):
    self.order_id = int(order_id)
    self.customer_id = int(customer_id)
    self.amount = float(amount)
    self.status = status.strip().lower()

    if self.amount < 0:
      raise ValueError(f"Order {self.order_id} has negative amount: {self.amount}")

  def __repr__(self):
    return (f"Order(order_id={self.order_id}, "
            f"customer_id={self.customer_id}, "
            f"amount={self.amount}, "
            f"status='{self.status}')")

  def is_high_value(self):
      return self.amount > 10000

  def is_valid(self):
      return self.status in self.VALID_STATUSES

orders = [
    Order(101, 201, 15000, "Pending"),
    Order(102, 202, 5000, "delivered"),
    Order(103, 203, 8000, "invalid_status")
    #Order(104, 204, -500, "pending")
]

for o in orders:
  print(o)
  print("High Value", o.is_high_value())
  print("Validity", o.is_valid())

Order(order_id=101, customer_id=201, amount=15000.0, status='pending')
High Value True
Validity True
Order(order_id=102, customer_id=202, amount=5000.0, status='delivered')
High Value False
Validity True
Order(order_id=103, customer_id=203, amount=8000.0, status='invalid_status')
High Value False
Validity False


In [ ]:
class Order:

    def __init__(self, order_id, customer_id, amount, status):
      self.order_id = int(order_id)
      self.customer_id = int(customer_id)
      self.amount = float(amount)
      self.status = status.strip().lower()

    def is_high_value(self):
        return self.amount > 10000

    def is_valid(self):
        valid_statuses = {"pending", "shipped", "delivered", "cancelled"}
        return self.status in valid_statuses

    def apply_discount(self, percent):
        return self.amount * (1 - percent / 100)


order1 = Order(101, 201, 15000, "pending")
order2 = Order(102, 202, 4000,  "invalid")

print(order1.is_high_value())       # True
print(order2.is_high_value())       # False
print(order1.is_valid())            # True
print(order2.is_valid())            # False
print(order1.apply_discount(10))    # 13500.0

True
False
True
False
13500.0


In [ ]:
class Order:

    def __init__(self, order_id, customer_id, amount, status):
        self.order_id    = order_id
        self.customer_id = customer_id
        self.amount      = amount
        self.status      = status

    def __repr__(self):
        return (f"Order("
                f"id={self.order_id}, "
                f"customer={self.customer_id}, "
                f"amount={self.amount}, "
                f"status='{self.status}')")

    def is_high_value(self):
        return self.amount > 10000

    def is_valid(self):
        valid_statuses = {"pending", "shipped", "delivered", "cancelled"}
        return self.status in valid_statuses

order1 = Order(101, 201, 15000, "pending")
print(order1)
# Order(id=101, customer=201, amount=15000, status='pending')

Order(id=101, customer=201, amount=15000, status='pending')


**Class variables — shared across all objects**

So far every variable lived on the object — different for every instance. Sometimes you want something shared across all objects. Like the list of valid statuses — that never changes per order, it is the same for every order in the system.

In [ ]:
class Order:

    VALID_STATUSES = {"pending", "shipped", "delivered", "cancelled"}
    HIGH_VALUE_THRESHOLD = 10000

    def __init__(self, order_id, customer_id, amount, status):
        self.order_id    = order_id
        self.customer_id = customer_id
        self.amount      = amount
        self.status      = status.strip().lower()

    def __repr__(self):
        return (f"Order("
                f"id={self.order_id}, "
                f"customer={self.customer_id}, "
                f"amount={self.amount}, "
                f"status='{self.status}')")

    def is_high_value(self):
        return self.amount > Order.HIGH_VALUE_THRESHOLD

    def is_valid(self):
        return self.status in Order.VALID_STATUSES

orders = [
    Order(101, 201, 15000, "Pending"),
    Order(102, 202, 4000,  "SHIPPED"),
    Order(103, 203, 25000, "invalid")
]

for order in orders:
    print(order)
    print(f"  High value : {order.is_high_value()}")
    print(f"  Valid      : {order.is_valid()}")
    print()

Order(id=101, customer=201, amount=15000, status='pending')
  High value : True
  Valid      : True

Order(id=102, customer=202, amount=4000, status='shipped')
  High value : False
  Valid      : True

Order(id=103, customer=203, amount=25000, status='invalid')
  High value : True
  Valid      : False



**Inheritance — building on top of something that already exists**

This is where OOP gets powerful for DE.
You are going to build three pipelines — one for orders, one for customers, one for products. All three need the same things — a name, a way to log messages, and a way to handle errors. Without inheritance you write that three times.
Inheritance lets you write it once in a base class and have all three pipelines receive it automatically.

Without using Inheritance.

In [2]:
class OrderPipeline:
    def __init__(self):
        self.name   = "OrderPipeline"
        self.errors = 0

    def log(self, message):
        print(f"[{self.name}] {message}")

    def summary(self):
        print(f"[{self.name}] Errors: {self.errors}")


class CustomerPipeline:
    def __init__(self):
        self.name   = "CustomerPipeline"
        self.errors = 0

    def log(self, message):
        print(f"[{self.name}] {message}")

    def summary(self):
        print(f"[{self.name}] Errors: {self.errors}")


class ProductPipeline:
    def __init__(self):
        self.name   = "ProductPipeline"
        self.errors = 0

    def log(self, message):
        print(f"[{self.name}] {message}")

    def summary(self):
        print(f"[{self.name}] Errors: {self.errors}")

order_pipeline = OrderPipeline()
customer_pipeline = CustomerPipeline()
product_pipeline = ProductPipeline()

order_pipeline.log("Order processed")
customer_pipeline.log("Customer validated")
product_pipeline.log("Product loaded")

order_pipeline.errors += 2
customer_pipeline.errors += 1

order_pipeline.summary()
customer_pipeline.summary()
product_pipeline.summary()


[OrderPipeline] Order processed
[CustomerPipeline] Customer validated
[ProductPipeline] Product loaded
[OrderPipeline] Errors: 2
[CustomerPipeline] Errors: 1
[ProductPipeline] Errors: 0


**The inheritance solution**

Write the common stuff once in a parent class. Children inherit it automatically.

**Step 1 — write the parent**


In [9]:
class BasePipeline:
  def __init__(self, name):
    self.name = name
    self.error = 1

  def log(self, message):
    print(f"[{self.name}] {message}")

  def summary(self):
    print(f"[{self.name}] Errors: {self.error}")

**Step 2 — write a child that inherits from it**


In [11]:
class OrderPipeline(BasePipeline):

  def __init__(self):
    super().__init__("OrderPipeline")

  def run(self, orders):
    self.log("Starting order pipeline")

    for order in orders:
      self.log(f"Processing order {order['order_id']}")

    self.log("Finished")
    self.summary()

pipeline = OrderPipeline()
pipeline.log("Starting")
pipeline.log("Processing records")
pipeline.summary()

[OrderPipeline] Starting
[OrderPipeline] Processing records
[OrderPipeline] Errors: 1


In [12]:
class CustomerPipeline(BasePipeline):

  def __init__(self):
    super().__init__("CustomerPipeline")

  def run(self, customers):
    self.log("Starting customer pipeline")

    for customer in customers:
        self.log(f"Processing customer {customer['name']}")

    self.log("Finished")
    self.summary()

In [13]:
orders = [
    {"order_id": 101, "amount": 15000},
    {"order_id": 102, "amount": 4000},
]

customers = [
    {"name": "Priya Mehta"},
    {"name": "Arjun Patel"},
]

op = OrderPipeline()
op.run(orders)

print()

cp = CustomerPipeline()
cp.run(customers)

[OrderPipeline] Starting order pipeline
[OrderPipeline] Processing order 101
[OrderPipeline] Processing order 102
[OrderPipeline] Finished
[OrderPipeline] Errors: 1

[CustomerPipeline] Starting customer pipeline
[CustomerPipeline] Processing customer Priya Mehta
[CustomerPipeline] Processing customer Arjun Patel
[CustomerPipeline] Finished
[CustomerPipeline] Errors: 1


**Overriding — when a child wants to do something differently**

Sometimes a child needs a slightly different version of the parent's method.

It can override it — replace the parent's version with its own:

In [15]:
class BasePipeline:

  def __init__(self, name):
    self.name   = name
    self.errors = 0

  def log(self, message):
    print(f"[{self.name}] {message}")

  def summary(self):
    print(f"[{self.name}] Errors: {self.errors}")

class OrderPipeline(BasePipeline):

  def __init__(self):
    super().__init__("OrderPipeline")
    self.high_value_count = 0

  def run(self, orders):
    self.log("Starting")
    for order in orders:
      if order["amount"] > 100000:
        self.log(f"Processed Order {order['order_id']}")\


    self.summary()

  def summary(self):
    super().summary()
    print(f"[{self.name}] High value orders: {self.high_value_count}")

orders = [
    {"order_id": 101, "amount": 15000},
    {"order_id": 102, "amount": 4000},
    {"order_id": 103, "amount": 22000},
]

op = OrderPipeline()
op.run(orders)


[OrderPipeline] Starting
[OrderPipeline] Errors: 0
[OrderPipeline] High value orders: 0


**Inheritance with Mobile Example**

In [17]:
class Phone:

  def __init__ (self, brand, battery):
    self.brand = brand
    self.battery = battery

  def make_call(self, number):
    print(f"[{self.brand}] {number}")

  def charge(self, amount):
    self.battery += amount
    print(f"[{self.brand}] battery is now {self.battery}%")

class SamsungPhone(Phone):

  def __init__(self, battery):
    super().__init__("Samsung", battery)
    self.camera_mode = "normal"
  def switch_camera(self, mode):
    self.camera_mode = mode
    print(f"Samsung camera switched to {self.camera_mode}")

iphone  = Phone("iPhone", 20)
samsung = Phone("Samsung", 45)

iphone.make_call("9876543210")
samsung.make_call("9123456789")

iphone.charge(30)
samsung.charge(10)

print(iphone.battery)    # 50
print(samsung.battery)   # 55


[iPhone] 9876543210
[Samsung] 9123456789
[iPhone] battery is now 50%
[Samsung] battery is now 55%
50
55


In [18]:
class SamsungPhone(Phone):

  def __init__(self, battery):
    super().__init__("Samsung", battery)
    self.camera_mode = "normal"

  def switch_camera(self, mode):
    self.camera_mode = mode
    print(f"Samsung camera switched to {self.camera_mode}")


s = SamsungPhone(40)

s.make_call("9876543210")
s.charge(20)
s.switch_camera("pro")

print(s.brand)        # Samsung
print(s.battery)      # 60
print(s.camera_mode)  # pro

[Samsung] 9876543210
[Samsung] battery is now 60%
Samsung camera switched to pro
Samsung
60
pro


In [22]:
class IPhone(Phone):

  def __init__(self, battery):
    super().__init__("iPhone", battery)
    self.face_id_enable = False

  def enabled_face_id(self):
    self.face_id_enabled = True
    print(f"iPhone FaceID enabled")

  def make_call(self, number):
    if self.enabled_face_id:
      print(f"iPhone verified your face — calling {number}")
    else:
      print(f"iPhone calling {number} without verification")

i = IPhone(80)

i.make_call("9876543210")

i.enabled_face_id()

i.make_call("9876543210")

i.charge(10)

print(i.battery)

iPhone verified your face — calling 9876543210
iPhone FaceID enabled
iPhone verified your face — calling 9876543210
[iPhone] battery is now 90%
90


**Step 4 — Now all three together**

In [23]:
phones = [
    Phone("Nokia", 90),
    SamsungPhone(40),
    IPhone(80),
]

for phone in phones:
    phone.make_call("9999999999")

[Nokia] 9999999999
[Samsung] 9999999999
iPhone verified your face — calling 9999999999


**Section 1.7 — Generators**

The problem first
You have a file with 10 million order records. You need to process all of them.
The naive approach:

In [24]:
def read_all_orders():
    orders = []
    for i in range(10_000_000):
        orders.append({"order_id": i, "amount": i * 10})
    return orders

all_orders = read_all_orders()

**The generator solution**

A generator does not produce everything at once. It produces one item, hands it to you, waits, then produces the next one. Only one item exists in memory at any moment.
Same 10 million orders — but now with a generator:

In [30]:
def generate_orders():
  for i in range(10_000_000):
    yield {"order_id": i, "amount": i * 10}
orders = generate_orders()
print(orders)


<generator object generate_orders at 0x7d329c6adf20>


In [31]:
print(next(orders))
print(next(orders))
print(next(orders))

{'order_id': 0, 'amount': 0}
{'order_id': 1, 'amount': 10}
{'order_id': 2, 'amount': 20}


**The real way to use generators — with a for loop**

In [32]:
def generate_orders(n):
  for i in range(n):
    yield{"order_id": i, "amount": i * 100}
for order in generate_orders(5):
  print(order)

{'order_id': 0, 'amount': 0}
{'order_id': 1, 'amount': 100}
{'order_id': 2, 'amount': 200}
{'order_id': 3, 'amount': 300}
{'order_id': 4, 'amount': 400}


**Write a generator function called valid_orders that:**

valid_orders(orders) should:
yield only orders where status is valid
valid statuses: pending, shipped, delivered, cancelled
skip invalid ones silently

Then use it to calculate total revenue
of only valid orders

In [38]:
orders = [
    {"order_id": 101, "amount": 15000, "status": "pending"},
    {"order_id": 102, "amount": 4000,  "status": "invalid"},
    {"order_id": 103, "amount": 8000,  "status": "shipped"},
    {"order_id": 104, "amount": 22000, "status": "bad_status"},
    {"order_id": 105, "amount": 3000,  "status": "delivered"},
]

def valid_orders(orders):
    valid_statuses = {"pending", "shipped", "delivered", "cancelled"}

    for order in orders:
        if order.get("status") in valid_statuses:
            yield order

total = sum(order["amount"] for order in valid_orders(orders))
print(total)


26000


**Section 1.8 — Production Python**

Logging — why print does not exist in production

Right now you use print() to see what your code is doing. In production that does not work for three reasons.

First — print output disappears. There is no record of what happened. When your pipeline fails at 3am you have nothing to look back at.

Second — print has no levels. You cannot distinguish between "this is just info" and "this is a critical error."

Third — print cannot be routed to files, monitoring systems, or alerting tools.

Logging solves all three. Run this:

In [39]:
import logging

logging.basicConfig(
    level  = logging.DEBUG,
    format = "%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)

logger.debug("Detailed info — only useful when debugging")
logger.info("Pipeline started — normal operations")
logger.warning("Something unexpected — pipeline still running")
logger.error("Something failed — needs attention")
logger.critical("Pipeline is down — immediate action needed")

ERROR:__main__:Something failed — needs attention
CRITICAL:__main__:Pipeline is down — immediate action needed
